# Title

## Module Imports

In [16]:
from dataclasses import dataclass
from typing import Callable
import pickle
import mlflow
import nico2_lib as n2l
import matplotlib.pyplot as plt
import numpy as np
from numpy import number
from numpy.typing import NDArray
import polars as pl
import seaborn as sns
from experiment import (
    CelltypeDimension,
    ExperimentResult,
    ModelDimension,
    ResultRecord,
    SampleDimension
)

In [3]:
MetricFn = Callable[[NDArray[number], NDArray[number]], float]


def build_reconstruction_metric_registry() -> dict[str, MetricFn]:
    return {"pearsonr": n2l.mt.pearson_metric, "spearmanr": n2l.mt.spearman_metric}


def _score_celltype(
    metric: MetricFn, observed: np.ndarray, predicted: np.ndarray
) -> float:
    return float(
        np.nan_to_num(
            np.array(
                [
                    metric(observed_cell, predicted_cell)
                    for observed_cell, predicted_cell in zip(observed, predicted)
                ]
            )
        ).mean()
    )

In [4]:
benchmark_experiment = mlflow.get_experiment_by_name("Default")
runs = mlflow.search_runs(experiment_ids=[benchmark_experiment.experiment_id])
last_run_id = runs.sort_values("start_time", ascending=False)["run_id"].iloc[0]

2026/02/20 14:53:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/20 14:53:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/20 14:53:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/20 14:53:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/20 14:53:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/20 14:53:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/20 14:53:30 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/20 14:53:30 INFO alembic.runtime.migration: Will assume non-transactional DDL.


In [6]:
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,params.n_samples,params.cache_dir,params.n_pca_components,params.seed,params.sample_length,params.data_dir,params.predictor_keys,params.dataloader_keys,tags.mlflow.source.git.commit,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user
0,6cb57c9675ef41ca93235b86c11ae74d,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 13:52:27.348000+00:00,2026-02-20 13:52:36.482000+00:00,10,../cache,10,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],7387332b6a7b99665d068a8dbf48737a0b13ee87,experiment.py,LOCAL,sedate-elk-649,egerc
1,e3357789bc4e4b98b8c3f6c2ce0182c1,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 12:28:11.819000+00:00,2026-02-20 12:28:20.931000+00:00,10,../cache,10,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],f609e47d2d869dd72a75a142382d17af98608819,experiment.py,LOCAL,luminous-pug-166,egerc
2,4503cec65cbe43e98a15188bac845c25,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 11:32:31.465000+00:00,2026-02-20 11:32:40.858000+00:00,10,../cache,25,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],f609e47d2d869dd72a75a142382d17af98608819,experiment.py,LOCAL,treasured-cow-4,egerc
3,ab49985ef55c4706b6d12413d52e9212,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 10:59:33.911000+00:00,2026-02-20 10:59:43.737000+00:00,10,../cache,25,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],f609e47d2d869dd72a75a142382d17af98608819,experiment.py,LOCAL,omniscient-cod-734,egerc
4,117782d4562a417ea8e80a1301d90e69,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 10:44:58.097000+00:00,2026-02-20 10:45:08.082000+00:00,10,../cache,25,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],f609e47d2d869dd72a75a142382d17af98608819,experiment.py,LOCAL,rogue-crab-626,egerc
5,5f060bcc4e8c444fa138cbcaa870af7c,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 10:07:02.603000+00:00,2026-02-20 10:07:12.565000+00:00,10,../cache,25,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],f609e47d2d869dd72a75a142382d17af98608819,experiment.py,LOCAL,abundant-hound-212,egerc
6,ca8af6c4a03f4323a78cfc95f474c07a,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-20 09:28:47.362000+00:00,2026-02-20 09:28:58.236000+00:00,10,../cache,25,0,20,../data,"['NMF_3', 'NMF_8']",['small_mouse_intestine_spatial'],f609e47d2d869dd72a75a142382d17af98608819,experiment.py,LOCAL,youthful-chimp-805,egerc
7,9a80e2c1ab3543c4a3219e68afd7a3cc,0,FINISHED,/Users/egerc/Documents/Projects/notebook_repos...,2026-02-19 16:07:39.844000+00:00,2026-02-19 16:07:50.626000+00:00,10,../cache,25,0,20,../data,None,None,e981dd74cd5a7049d1eb276f5fc78ad72d1f882a,experiment.py,LOCAL,indecisive-fish-799,egerc


In [17]:
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=last_run_id, artifact_path="results.pkl"
)
with open(artifact_path, "rb") as f:
    results = pickle.load(f)

In [28]:
results[0].results[0].celltype_id

0